# dataset: loading square crops (M3)

Loads the square crops written by `capture.ipynb` (format in `docs/contracts.md`):
`<crops_root>/<session_id>/manifest.csv` plus `squares/*.png`.
**Splits are always by session_id, never by frame**, so the test set really contains
unseen lighting / camera placements.

torch is imported lazily, so everything except `build_transforms` and the torch Dataset
works without it.

Milestone: M3 (square classifiers).

**TODO** (see `docs/M3_TODO.md`)
- [ ] `list_sessions`, `load_manifest`, `split_by_session`, `read_crop`
- [ ] `SquareCropDataset`, `class_weights`
- [ ] `build_transforms`: settle the augmentation set (flips, perspective, blur).
- [ ] `load_chessred`: ChessReD loader (optional pretraining data).
- [ ] `load_chesscog_synthetic`: chesscog synthetic loader (optional pretraining data).
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

%run ./labels.ipynb

In [ ]:
MANIFEST_COLUMNS = ("session_id", "game_id", "move_idx", "frame_id", "square", "label", "path", "fen")

## Manifests
`load_manifest` concatenates the sessions' manifests and adds the training targets:
`y` (13-class index), `y_occ` (0/1) and `y_color3` (0 empty / 1 white / 2 black), plus
`abs_path` (crop path resolved against the session directory).

In [ ]:
def list_sessions(crops_root: str | Path) -> list[str]:
    """Session ids under crops_root (directories that contain a manifest.csv), sorted."""
    # TODO: implement (Path.glob("*/manifest.csv"))
    raise NotImplementedError


def load_manifest(crops_root: str | Path, sessions: Iterable[str] | None = None) -> pd.DataFrame:
    """Concatenate manifest.csv of the given sessions (all sessions if None).

    Adds columns: `abs_path` (crop path resolved against the session dir), `y` (13-class
    index), `y_occ` (0/1), `y_color3` (0 empty / 1 white / 2 black).
    """
    # Design:
    # - pd.read_csv each session's manifest; read frame_id / game_id / session_id as str
    #   (otherwise "000001" becomes 1).
    # - Fail loudly: no sessions -> ValueError, missing manifest -> FileNotFoundError,
    #   missing MANIFEST_COLUMNS or labels not in CLASSES -> ValueError. Bad data found here
    #   is much cheaper than bad data found after a training run.
    # - Targets via CLASS_TO_INDEX, label_to_occupancy, label_to_color3 (labels.ipynb).
    # TODO: implement
    raise NotImplementedError


def split_by_session(cfg) -> dict[str, pd.DataFrame]:
    """Return {"train", "val", "test"} DataFrames from cfg.data.*_sessions.

    Empty session lists give empty DataFrames. Raises if a session is in two splits.
    """
    # Design:
    # - Check pairwise overlap of the three session lists first (ValueError naming both splits).
    # - load_manifest per non-empty list; empty list -> DataFrame with MANIFEST_COLUMNS.
    # TODO: implement
    raise NotImplementedError


def read_crop(path: str | Path) -> np.ndarray:
    """Read a crop as an RGB uint8 array (H, W, 3)."""
    # Design: cv2.imread gives BGR (or None if the file is missing -> FileNotFoundError);
    # convert to RGB because torchvision / ImageNet normalization assume RGB.
    # TODO: implement
    raise NotImplementedError

## torch side
Transforms come from `cfg.cnn.input_size` and `cfg.cnn.augment`. Eval: resize -> tensor ->
ImageNet normalize; train adds augmentation. `SquareCropDataset` falls back to a plain
`object` base class when torch is missing so this notebook still runs.

In [ ]:
def build_transforms(cfg, train: bool):
    """torchvision transform for crops, from cfg.cnn.input_size and cfg.cnn.augment.

    Eval: resize -> tensor -> ImageNet normalize. Train adds augmentation.
    """
    # Design:
    # - Input is an RGB uint8 numpy array -> T.ToPILImage first, T.Resize((h, w)).
    # - Train only: ColorJitter (cfg.cnn.augment.color_jitter), small RandomAffine
    #   (rotation_deg, translate) to mimic session differences.
    # - Always end with T.ToTensor and ImageNet mean/std normalization (pretrained backbone).
    # Decisions for you: horizontal flips are label-safe for a single piece; perspective / blur?
    # TODO: implement
    raise NotImplementedError


def _torch_dataset_base():
    # Plumbing (kept implemented): the class statement below calls this at definition time,
    # so the notebook must work without torch.
    try:
        from torch.utils.data import Dataset
    except ImportError:  # notebook must run without torch
        return object
    return Dataset


class SquareCropDataset(_torch_dataset_base()):
    """torch Dataset over a manifest DataFrame. Item: (image tensor, class index).

    `target` picks the label column: "y" (13 classes), "y_occ" or "y_color3".
    """

    def __init__(self, df: pd.DataFrame, transform=None, target: str = "y"):
        # Design: keep plain Python lists of abs_path and int targets (cheap to pickle into
        # DataLoader workers), plus the transform.
        # TODO: implement
        raise NotImplementedError

    def __len__(self) -> int:
        # TODO: implement
        raise NotImplementedError

    def __getitem__(self, i: int):
        # Design: read_crop, apply transform if any, return (image, target).
        # TODO: implement
        raise NotImplementedError


def class_weights(df: pd.DataFrame) -> np.ndarray:
    """Inverse-frequency weights (13,) for the loss; empty squares dominate real data.

    Classes absent from df get weight 0.
    """
    # Design: counts via np.bincount(df["y"], minlength=13); w_c = N / (n_present_classes * n_c)
    # for present classes so the weighted average stays ~1.
    # TODO: implement
    raise NotImplementedError

## External datasets (optional pretraining)

In [ ]:
def load_chessred(root: str | Path, split: str = "train") -> pd.DataFrame:
    """ChessReD (Masouris & van Gemert 2023): real photos of full boards.

    Source format: `images/<game>/*.jpg` and one `annotations.json` (COCO-like) with per-image
    piece annotations (category + chessboard_position like "e4") and, for a subset, the four
    board corners. Splits are given in the JSON.
    """
    # Design:
    # - For images with corner annotations, warp with homography.ipynb (warp_board) and cut
    #   crops with the same geometry as capture.ipynb (extract_all_crops).
    # - Return a DataFrame with the manifest columns (session_id = "chessred_<game>") so it
    #   plugs into SquareCropDataset. Map ChessReD category names to CLASSES.
    # - Probably cache crops to disk once rather than cropping on the fly.
    # TODO: implement
    raise NotImplementedError


def load_chesscog_synthetic(root: str | Path, split: str = "train") -> pd.DataFrame:
    """chesscog synthetic dataset (Wölflein & Arandjelović 2021): Blender renders.

    Source format: `<split>/NNNN.png` with a matching `NNNN.json` holding `fen`, `corners`
    (4 board corners in pixels), `white_turn` and lighting metadata.
    """
    # Design:
    # - Warp each render with its corners, crop the 64 squares like capture.ipynb, derive
    #   labels from the FEN via board_to_labels.
    # - Return a manifest-style DataFrame (session_id = "synth"). Used when
    #   cfg.cnn.pretrain_on == "synthetic".
    # TODO: implement
    raise NotImplementedError

## Self-check
(was tests/test_dataset.py). Write fake sessions to a temporary directory.

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# Helper to write: a fake session = a manifest.csv with 64 rows per frame for a given board
# (labels from board_to_labels), crops not written. Use tempfile.TemporaryDirectory.
# - load_manifest on a 2-frame session: 128 rows, frame_id stays "000000" (str);
#   e1 row: label "K", y == CLASS_TO_INDEX["K"], y_occ 1, y_color3 1; e8 has y_color3 2;
#   empty rows have y 0; abs_path ends with "s01/squares/000000_e1.png".
# - class_weights: shape (13,), weight(empty) < weight(K).
# - list_sessions on sessions "s02", "s01" -> ["s01", "s02"]; load_manifest(root) loads both.
# - A label "X" in the manifest -> ValueError("unknown labels..."); missing session ->
#   FileNotFoundError.
# - split_by_session with train [a, b], val [c]: right sessions per split, empty test;
#   the same session in train and val -> ValueError mentioning "both".
# - With torch + cv2: write 3 blank (200, 100, 3) PNGs, SquareCropDataset(df)[0] returns
#   (image of shape (200, 100, 3), its y) and len == 3.
# - build_transforms(cfg, train=False) on a (200, 100, 3) uint8 crop -> float tensor
#   (3, H, W) with H, W = cfg.cnn.input_size; eval is deterministic, train is random.
# - load_chessred / load_chesscog_synthetic on tiny fixtures -> manifest columns, labels in
#   CLASSES, 64 rows per board.
print("dataset: self-check not written yet (see plan above)")